# Day 1 | Notebook 3: RedisVL and Production Vector Databases
**Author: Sattaya Singkul**

Now that we deeply understand the math and the limitations of FAISS, let's deploy the industry standard.
Redis is not only an in-memory cache, but an extremely fast, natively integrated Vector Database. We'll interface with it using Python **RedisVL** (Redis Vector Library).

### Running Docker Environment
Before running this code, ensure Redis Stack is running on your system via docker (See the `docker/` folder in your project directory).


In [3]:
!pip install redisvl pandas sentence-transformers

## Step 1: Connecting and RedisVL Architecture
RedisVL is designed exclusively to abstract away raw Redis commands, providing simple Python objects to manage schemas, indices, and querying pipelines.


In [18]:
from redisvl.index import SearchIndex
import pandas as pd

# Connect to our dockerized Redis using its SERVICE NAME (Standard Docker Design)
redis_url = "redis://redis-vector-db:6379"
print(f"Connecting to Internal Docker Network at {redis_url}")


Connecting to Internal Docker Network at redis://redis-vector-db:6379


## Step 2: Complex Schema Definition and Indexing (HNSW)
Unlike FAISS where we only fed it raw numpy floats, Redis mixes structured metadata (SQL-style) with unstructured embeddings.
We'll define an e-commerce catalog dataset using a hybrid Schema!


In [21]:
from redisvl.schema import IndexSchema

schema_dict = {
    "index": {
        "name": "ecommerce_catalog",
        "prefix": "item:", 
        "storage_type": "json"  # <--- Change this from 'hash' (or default) to 'json'
    },
    "fields": [
        {"name": "product_id", "type": "tag"},       # Exact match queries
        {"name": "category", "type": "tag"},         # Filtering
        {"name": "brand", "type": "tag"},            # Filtering
        {"name": "price", "type": "numeric"},        # Range filtering
        {"name": "description", "type": "text"},     # Full text search fallback
        {"name": "location", "type": "geo"},         # <--- NEW: Geo-Spatial field for local discovery
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": 384,                     # Dimension count matching our HuggingFace model
                "distance_metric": "cosine",     # Best for semantic NLP descriptions
                "algorithm": "hnsw",             # HNSW Graph!
                "datatype": "float32",
                "m": 16,                         # Standard HNSW link density
                "ef_construction": 200
            }
        }
    ]
}

schema = {
    "index": {
        "name": "product_index",
        "prefix": "product",
        "storage_type": "json"  # <--- Change this from 'hash' (or default) to 'json'
    },
    "fields": [
        # ... your other fields ...
        {
            "name": "embedding", 
            "type": "vector", 
            "attrs": {
                "dims": 384, # all-MiniLM-L6-v2 uses 384 dimensions
                "distance_metric": "cosine", 
                "algorithm": "flat"
            }
        }
    ]
}

from redisvl.index import SearchIndex
index = SearchIndex.from_dict(schema)

schema = IndexSchema.from_dict(schema_dict)

index = SearchIndex(schema, redis_url=redis_url)
index.create(overwrite=True) 

print("Index successfully provisioned in Redis!")
print("Go to http://localhost:8001 -> 'Browser' to see the newly created graph.")


Index successfully provisioned in Redis!
Go to http://localhost:8001 -> 'Browser' to see the newly created graph.


In [22]:
# from redisvl.index import SearchIndex
# index = SearchIndex.from_dict(schema)
# index

In [23]:
# index = IndexSchema.from_dict(schema_dict)
# index

## Step 3: Generating Data with sentence-transformers
To make this realistic, we'll generate real 384-dimensional dense vectors using a lightweight HuggingFace transformer model.


In [24]:
from sentence_transformers import SentenceTransformer
import time
import numpy as np

print("Loading local embedder model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

raw_products = [
    {"product_id": "101", "category": "laptops", "brand": "Apple", "price": 1200.0, "location": "-122.4194,37.7749", "description": "MacBook Air M2 with 16GB RAM for creative professionals"}, # SF
    {"product_id": "102", "category": "laptops", "brand": "Dell", "price": 850.0, "location": "-122.3321,47.6062", "description": "Dell XPS 13 for business professionals, lightweight"},  # Seattle
    {"product_id": "103", "category": "accessories", "brand": "Logitech", "price": 50.0, "location": "-74.0060,40.7128", "description": "Ergonomic wireless mouse with fast scrolling"},       # NY
    {"product_id": "104", "category": "audio", "brand": "Sony", "price": 350.0, "location": "-122.4194,37.7749", "description": "Sony WH-1000XM5 noise cancelling over-ear headphones"},  # SF
    {"product_id": "105", "category": "accessories", "brand": "Apple", "price": 150.0, "location": "-74.0060,40.7128", "description": "Magic keyboard with numeric keypad"},               # NY
    {"product_id": "106", "category": "audio", "brand": "Apple", "price": 250.0, "location": "-118.2437,34.0522", "description": "Airpods Pro gen 2 wireless earbuds"},                # LA
]

docs_to_load = []
for p in raw_products:
    vector = embedder.encode(p["description"]).tolist() # convert numpy to byte for Redis
    # vector = embedder.encode(p["description"]).astype(np.float32).tobytes()
    doc = p.copy()
    doc["embedding"] = vector
    docs_to_load.append(doc)

start = time.time()
keys = index.load(docs_to_load)
dur = time.time() - start

print(f"Loaded {len(keys)} document vectors into Redis in {dur:.3f} seconds.")


Loading local embedder model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 6 document vectors into Redis in 0.012 seconds.


## Step 4: Database Inspection & Visualizers
As a professional engineer, you must verify what is physically inside the database. Unlike a SQL table, Redis stores Vectors as binary blobs. Let's inspect the "Raw" data first, then use a "High-Level" Pandas visualizer.

### 4.1 Raw Inspection (The Deep Dive)
Notice how the `embedding` field looks like random characters? That is the 384-dimensional float array compressed into bytes!


In [25]:
keys = index.load(docs_to_load)
keys

['item:01KPZBGNQ0TD18E6JNSEV9DRHD',
 'item:01KPZBGNQ0TD18E6JNSEV9DRHE',
 'item:01KPZBGNQ0TD18E6JNSEV9DRHF',
 'item:01KPZBGNQ0TD18E6JNSEV9DRHG',
 'item:01KPZBGNQ0TD18E6JNSEV9DRHH',
 'item:01KPZBGNQ0TD18E6JNSEV9DRHJ']

In [26]:
import json

# Use the underlying redis client
client = index.client

# Grab the actual key that RedisVL generated for the first product (Apple MacBook)
actual_key = keys[0] 
print(f"Fetching dynamically generated key: {actual_key}")

# Fetch the JSON document using that exact key
raw_item = client.json().get(actual_key)

print(f"\n--- Raw Redis Object ({actual_key}) ---")
print(f"Brand: {raw_item['brand']}")
print(f"Price: {raw_item['price']}")
print(f"Vector Preview (First 3 elements): {raw_item['embedding'][:3]}...")

Fetching dynamically generated key: item:01KPZBGNQ0TD18E6JNSEV9DRHD

--- Raw Redis Object (item:01KPZBGNQ0TD18E6JNSEV9DRHD) ---
Brand: Apple
Price: 1200.0
Vector Preview (First 3 elements): [0.04408327117562294, 0.021286752074956897, -0.0013360597658902409]...


### 4.2 High-Level Visualizer (Pandas)
Let's see the entire database state as a clean table.


In [27]:
import pandas as pd
from IPython.display import display

# Use json().get() instead of hgetall()
data = [client.json().get(k) for k in keys]

# No need to decode bytes to strings! RedisJSON already did it for us.
df = pd.DataFrame(data)

# We drop the heavy embedding for the visual table
if 'embedding' in df.columns:
    display(df.drop(columns=['embedding']))
else:
    display(df)

,product_id,category,brand,price,location,description
0,101,laptops,Apple,1200.0,"-122.4194,37.7749",MacBook Air M2 with 16GB RAM for creative prof...
1,102,laptops,Dell,850.0,"-122.3321,47.6062","Dell XPS 13 for business professionals, lightw..."
2,103,accessories,Logitech,50.0,"-74.0060,40.7128",Ergonomic wireless mouse with fast scrolling
3,104,audio,Sony,350.0,"-122.4194,37.7749",Sony WH-1000XM5 noise cancelling over-ear head...
4,105,accessories,Apple,150.0,"-74.0060,40.7128",Magic keyboard with numeric keypad
5,106,audio,Apple,250.0,"-118.2437,34.0522",Airpods Pro gen 2 wireless earbuds


## Step 5: Advanced Query Workflows
FAISS couldn't filter by price. Let's execute highly complex **Hybrid Searches** and **Radius Searches** using RedisVL!

### 5.1 Hybrid Search (Semantic + Metadata)
"Find an Apple or Logitech mouse under $100."


In [28]:
from redisvl.query import VectorQuery
from redisvl.query.filter import Tag, Num

user_query = "device"
query_emb = embedder.encode(user_query).tolist()

price_filter = Num("price") < 150
brand_filter = (Tag("brand") == "Apple") | (Tag("brand") == "Logitech")
hybrid_filter = price_filter & brand_filter

# 1. Create the query object first
hq = VectorQuery(
    vector=query_emb,
    vector_field_name="embedding",
    return_fields=["brand", "price", "description"],
    num_results=3
)

# 2. Apply the filter on a separate line!
hq.set_filter(hybrid_filter)

# 3. Now execute the query
results = index.query(hq)
for r in results:
    print(f"Match: [{r['brand']}] ${r['price']} - {r['description']}")


Match: [Logitech] $50 - Ergonomic wireless mouse with fast scrolling
Match: [Logitech] $50 - Ergonomic wireless mouse with fast scrolling


### 5.2 Radius Search (Discovery Architecture)
Integrating Knowledge from Foundation: Sometimes we don't want "Top-K". We want "Everything within 0.1 distance".


In [29]:
from redisvl.query import RangeQuery

# Find everything highly similar to Airpods
target_desc = "Airpods Pro gen 2 wireless earbuds"
target_emb = embedder.encode(target_desc).tolist()

# RangeQuery finds all vectors within a radius
rq = RangeQuery(
    vector=target_emb,
    vector_field_name="embedding",
    return_fields=["description"],
    distance_threshold=0.2, # Strict radius
    num_results=3
)

range_results = index.query(rq)
print(f"Found {len(range_results)} items in the 0.2 Similarity Radius:")
for r in range_results:
    print(f" -> {r['description']} (Dist: {r['vector_distance']})")

Found 2 items in the 0.2 Similarity Radius:
 -> Airpods Pro gen 2 wireless earbuds (Dist: -1.19209289551e-07)
 -> Airpods Pro gen 2 wireless earbuds (Dist: -1.19209289551e-07)


### 5.3 Query Processing - Under the Hood (PRO LAB)
To truly master RedisVL, you must understand exactly what happens between your Python function and the Redis database.

**The 3 Stages of a Vector Query:**
1. **Vectorization**: The input `str` is converted into a 384-Float array.
2. **Command Generation**: RedisVL translates your Python object into a raw `FT.SEARCH` string.
3. **Hydration**: Redis returns binary blobs, which RedisVL "hydrates" back into a readable dictionary.


In [30]:
# STAGE 1: The Input Transformation
query_text = "I need a fast laptop"
query_vector = embedder.encode(query_text).tolist()

print(f"Input: {query_text}")
print(f"Vector (First 5 dimensions): {query_vector[:5]}...")

# STAGE 2: The Command Generator
vq = VectorQuery(
    vector=query_vector,
    vector_field_name="embedding",
    return_fields=["brand"],
    num_results=1
)

print("\n--- Raw Redis Command Generated ---")
# RedisVL uses str() to output the actual FT.SEARCH command
print(str(vq))

# STAGE 3: The Result Hydration
raw_res = index.query(vq)
print("\n--- Hydrated Result Object ---")
print(raw_res[0])


Input: I need a fast laptop
Vector (First 5 dimensions): [-0.016252269968390465, 0.036395296454429626, 0.02309749275445938, -0.009679346345365047, 0.027826795354485512]...

--- Raw Redis Command Generated ---
*=>[KNN 1 @embedding $vector AS vector_distance] RETURN 2 brand vector_distance SORTBY vector_distance ASC DIALECT 2 LIMIT 0 1

--- Hydrated Result Object ---
{'id': 'item:01KPZBGNPFCJBRRG9JTZQDB1Z0', 'vector_distance': '0.475866019726', 'brand': 'Dell'}


## Step 6: Production Mastery - Semantic Caching
**The Concept:** Why pay to embed the exact same user query 1,000 times? 
If a user asks "Apple laptop" and 5 seconds later another user asks "Cheap Apple Macbook", we can return the cached result semantically!


In [31]:
import time
from redisvl.extensions.llmcache import SemanticCache
from redisvl.utils.vectorize import HFTextVectorizer

# 1. Initialize the vectorizer with your specific model
# This tells the cache to expect 384 dimensions instead of 768
hf_vectorizer = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Initialize Cache 
cache = SemanticCache(
    name="llm_cache",
    redis_url=redis_url,
    distance_threshold=0.25, 
    vectorizer=hf_vectorizer, # <--- Pass the vectorizer here!
    overwrite=True # <--- IMPORTANT: Drops the broken 768-dim index from your previous run
)

# 3. Simulate User 1 (The Initial Request)
q1 = "How much is a Macbook Air?"
ans1 = "The Macbook Air M2 is $1200."

# Look how clean this is! No more manual embedder.encode()
cache.store(prompt=q1, response=ans1)

# 4. Simulate User 2 (Semantic Near-Duplicate)
q2 = "Price for a Macbook Air M2?"
start = time.time()

# Just pass the raw text prompt, the cache vectorizes it automatically
hit = cache.check(prompt=q2)
duration = time.time() - start

if hit:
    print(f"✅ CACHE HIT in {duration:.4f}s!")
    print(f"Question: {q2}")
    print(f"Cached Answer: {hit[0]['response']}")
else:
    print("❌ CACHE MISS")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CACHE HIT in 0.0125s!
Question: Price for a Macbook Air M2?
Cached Answer: The Macbook Air M2 is $1200.


## Step 7: Data Management Lifecycle (Update / Delete)
Databases aren't static. In production, products go out of stock or update descriptions.


In [34]:
from redis.commands.json.path import Path

# UPDATE
index.load([{"product_id": "102", "category": "laptops", "brand": "HCU", "price": 799.0, "description": "Dell XPS 13 for business professionals, lightweight"}], id_field="product_id")

# Execute fetch
updated = client.json().get("item:102")
print(f"Updated XPS Price: ${updated['price']}")

# # DELETE
# client.delete("item:105")
# print("Deleted Magic Keyboard from Database.")


Updated XPS Price: $799.0


In [35]:
# DELETE
client.delete("item:102")

1

In [36]:
# delete index data + value data in DB
index.delete(drop=True)

In [37]:
# delete all data in DB
client.flushall()

True

---

# 🏆 BONUS MASTERY: Real-World Geo-Semantic Discovery
In production (eCommerce, Food Delivery, Real Estate), search isn't just about **what**, it's about **where**.
Redis allows us to atomsitically combine Vector Similarity with Geo-Spatial math.

### The Scenario: "Local Pick-up Only"
A user in **San Francisco** is looking for an "Apple keyboard" (Semantic). They need to pick it up today at a shop within 50 miles.

Let's execute the **Geo-Filtered Semantic Query**.


In [39]:
from redisvl.query.filter import Geo, GeoRadius

user_location = (-122.4194, 37.7749) # San Francisco
query_text = "Apple keyboard"
query_vector = embedder.encode(query_text).tolist()

# 1. Define the Geo Filter (Stay within 50 miles of SF)
geo_filter = Geo("location") == GeoRadius(user_location[0], user_location[1], 50, unit="mi")

# 2. Combine with Vector Query
vq = VectorQuery(
    vector=query_vector,
    vector_field_name="embedding",
    filter_expression=geo_filter,
    return_fields=["brand", "description", "location"],
    num_results=3
)

geo_results = index.query(vq)

print(f"--- Results for User in San Francisco ---")
if not geo_results:
    print("Zero local results found.")
for r in geo_results:
    print(f"Match: [{r['brand']}] {r['description']} at {r['location']}")

print("\n--- Testing the Isolation ---")
# If the same user moves to New York (-74.0060, 40.7128), can they see the SF stock?
ny_location = (-74.0060, 40.7128)
vq_ny = VectorQuery(
    vector=query_vector,
    vector_field_name="embedding",
    filter_expression=Geo("location") == GeoRadius(ny_location[0], ny_location[1], 50, unit="mi"),
    return_fields=["brand", "description"],
    num_results=3
)

ny_results = index.query(vq_ny)
print(f"Results for User in New York: {len(ny_results)} local items found.")


--- Results for User in San Francisco ---
Match: [Apple] MacBook Air M2 with 16GB RAM for creative professionals at -122.4194,37.7749
Match: [Apple] MacBook Air M2 with 16GB RAM for creative professionals at -122.4194,37.7749
Match: [Apple] MacBook Air M2 with 16GB RAM for creative professionals at -122.4194,37.7749

--- Testing the Isolation ---
Results for User in New York: 3 local items found.


### 🚀 Professional Tips for Hybrid Discovery
1. **The Index Intersection**: When you combine Geo + Vector, Redis doesn't just "filter" the vector results. It creates an intersection of the indices. This allows it to handle 100M+ items with millisecond latency.
2. **Precision Balance**: In many apps, it's better to use **Coarse Filtering** (e.g., zip codes) first as a `TAG` filter, and then only use the complex `GeoRadius` math for the final ranking to save CPU.
3. **Multi-Store Management**: In large enterprises, each store often has its own vector partition. Redis Cluster allows you to distribute these geographically across different global regions while maintaining a unified API.

---
### You have finished Notebook 3 and Day 1 Foundation.
Everything is set for Day 2: Building your first AI-Agent Application!
